In [25]:
import os
import glob
import gpxpy
import folium
from folium import Element

# 1️⃣ GPX folder
gpx_folder = "gpx_tracks"
gpx_files = glob.glob(os.path.join(gpx_folder, "*.gpx"))

colors = ["red","blue","green","purple","orange","darkred",
          "cadetblue","darkgreen","black"]

# 2️⃣ Base map
m = folium.Map(location=[0,0], zoom_start=2)

all_points = []

# 3️⃣ Mobile bottom info panel
panel_html = """
<div id="info-panel"
style="
position: fixed;
bottom: 0;
left: 0;
right: 0;
background: white;
padding: 12px;
border-top: 2px solid #ccc;
box-shadow: 0 -2px 6px rgba(0,0,0,0.3);
font-family: sans-serif;
display: none;
z-index: 9999;
">
<div style="display:flex;justify-content:space-between">
<b>Track Info</b>
<button onclick="document.getElementById('info-panel').style.display='none'">Close</button>
</div>
<div id="info-content"></div>
</div>

<script>
function showInfo(text){
    var panel = document.getElementById("info-panel");
    document.getElementById("info-content").innerHTML = text;
    panel.style.display = "block";
}
</script>
"""
m.get_root().html.add_child(Element(panel_html))

# 4️⃣ Function for emoji markers
def emoji_marker(map_obj, lat, lon, emoji, text=""):
    folium.Marker(
        [lat, lon],
        popup=text,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size:22px;
                transform: translate(-10px, -10px);
            ">
                {emoji}
            </div>
            """
        )
    ).add_to(map_obj)

# 5️⃣ Load GPX files
for i, file in enumerate(gpx_files):
    with open(file, "r") as f:
        gpx = gpxpy.parse(f)

    points = []
    distance = 0

    for track in gpx.tracks:
        for segment in track.segments:
            distance += segment.length_3d()
            for p in segment.points:
                coord = (p.latitude, p.longitude)
                points.append(coord)
                all_points.append(coord)

    if not points:
        continue

    name = os.path.basename(file)
    km = distance / 1000
    info_html = f"<b>{name}</b><br>Distance: {km:.2f} km"

    layer = folium.FeatureGroup(name=f"{name} ({km:.1f} km)")
    color = colors[i % len(colors)]

    polyline = folium.PolyLine(points, color=color, weight=5)
    polyline.add_to(layer)

    # click → show bottom info panel + highlight
    polyline.add_child(
        folium.Element(
            f"""
            <script>
            {polyline.get_name()}.on('click', function(e) {{
                showInfo(`{info_html}`);
                this.setStyle({{weight:8}});
            }});
            </script>
            """
        )
    )
    # start and end markers with distance
    emoji_marker(layer, points[0][0], points[0][1], "🟢", f"{name} start – {km:.2f} km")
    emoji_marker(layer, points[-1][0], points[-1][1], "🔴", f"{name} finish – {km:.2f} km")

    layer.add_to(m)
    
folium.Marker(
    location=[-37.34830885070061, 144.15402342110187],
    popup="<b>If lost<br>return here</b>",
    tooltip="Tap for info",
    icon=folium.Icon(color="green", icon="home")
).add_to(m)

folium.Marker(
    location=[-37.30992684492659, 144.13980049294116],
    popup="<b>Hepburn Spa",
    tooltip="Tap for info",
    icon=folium.Icon(color="blue", icon="info-sign")
).add_to(m)

folium.Marker(
    location=[-37.30992684492659, 144.13980049294116],
    popup="<b>Hepburn Spa",
    tooltip="Tap for info",
    icon=folium.Icon(color="blue", icon="info-sign")
).add_to(m)

folium.Marker(
    location=[-37.342009513860326, 144.1484603261187],
    popup="<b>The Convent",
    tooltip="Tap for info",
    icon=folium.Icon(color="blue", icon="info-sign")
).add_to(m)

folium.Marker(
    location=[-37.389862824286695, 144.11930658012355],
    popup="<b>Sailors Falls",
    tooltip="Tap for info",
    icon=folium.Icon(color="blue", icon="info-sign")
).add_to(m)

# 6️⃣ Auto zoom to all tracks
if all_points:
    m.fit_bounds(all_points)

# 7️⃣ Layer toggle
folium.LayerControl().add_to(m)

# 8️⃣ Current location – works on mobile & desktop
locate_js = f"""
<script>
window.onload = function(){{
    // get Folium map object
    var map = {m.get_name()};
    map.locate({{setView:true, maxZoom:16}});
    map.on('locationfound', function(e){{
        var radius = e.accuracy/2;
        // marker with emoji
        L.marker(e.latlng, {{
            icon: L.divIcon({{
                html: '📍',
                className: '',
                iconSize: [24,24]
            }})
        }}).addTo(map).bindPopup("You are here").openPopup();
        L.circle(e.latlng, radius).addTo(map);
    }});
    map.on('locationerror', function(e){{
        console.log("Location error: " + e.message);
    }});
}};
</script>
"""
m.get_root().html.add_child(Element(locate_js))

# 9️⃣ Save
m.save("gpx_viewer_mobile.html")
print("Map saved as gpx_viewer_mobile.html")

Map saved as gpx_viewer_mobile.html


In [ ]:
folium.Marker(
    location=[-37.34830885070061, 144.15402342110187],
    popup="<b>If lost<br>return here</b>",
    tooltip="Tap for info",
    icon=folium.Icon(color="red", icon="home")
).add_to(m)

folium.Marker(
    location=[-37.30992684492659, 144.13980049294116],
    popup="<b>Hepburn Spa",
    tooltip="Tap for info",
    icon=folium.Icon(color="red", icon="info-sign")
).add_to(m)